# 0) Imports

In [1]:
# IMPORTS
import numpy as np
import pandas as pd

#Fin Data Sources
import yfinance as yf
import pandas_datareader as pdr

#Data viz
import plotly.graph_objs as go
import plotly.express as px

import time
from datetime import date

import requests 

# 1) Let's go

## 1.

Which year had the highest number of additions (starting from 2020)?

Using the list of S&P 500 companies from Wikipedia's S&P 500 companies page, download the data including the year each company was added to the index.

In [2]:
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

# Fetch the HTML content with headers
response = requests.get(url, headers=headers)

Steps:

1. Create a DataFrame with company tickers, names, and the year they were added.
2. Extract the year from the addition date and calculate the number of stocks added each year.
3. Which (full) year had the highest number of additions, starting from 2020?

▎ Note: When stocks are added to the S&P 500, they usually experience a price bump as investors and index funds buy shares following the announcement.

In [3]:
from io import StringIO

sp500 = pd.read_html(StringIO(response.text))[0]
df = sp500[['Symbol', 'Security', 'Date added']].copy()
df['Date added'] = pd.to_datetime(df['Date added'], errors='coerce')
df['Year added'] = df['Date added'].dt.year

df.head()


,Symbol,Security,Date added,Year added
0,MMM,3M,1957-03-04,1957
1,AOS,A. O. Smith,2017-07-26,2017
2,ABT,Abbott Laboratories,1957-03-04,1957
3,ABBV,AbbVie,2012-12-31,2012
4,ACN,Accenture,2011-07-06,2011


In [4]:
additions_per_year = df['Year added'].value_counts().sort_index()
full_years = additions_per_year.loc[2020:2025]

full_years

Year added
2020    10
2021    10
2022    15
2023    15
2024    16
2025    18
Name: count, dtype: int64

## 2.

[Macro] Indexes YTD (as of 21 August 2026)
How many indexes (out of 10) have better year-to-date returns than the US (S&P 500) as of August 21, 2026?

Using Yahoo Finance World Indices data, compare the year-to-date (YTD) performance (1 January-21 August 2026, use the closing price to calculate the growth) of major stock market indexes for the following countries:

United States - S&P 500 (^GSPC)
China - Shanghai Composite (000001.SS)
Hong Kong - HANG SENG INDEX (^HSI)
Australia - S&P/ASX 200 (^AXJO)
India - Nifty 50 (^NSEI)
Canada - S&P/TSX Composite (^GSPTSE)
Germany - DAX (^GDAXI)
United Kingdom - FTSE 100 (^FTSE)
Japan - Nikkei 225 (^N225)
Mexico - IPC Mexico (^MXX)
Brazil - Ibovespa (^BVSP)

Hint: use start_date='2026-01-01' and end_date='2026-08-21' parameters when downloading daily data in yfinance

In [5]:
indexes = {
    '^GSPC': 'United States - S&P 500',
    '000001.SS': 'China - Shanghai Composite',
    '^HSI': 'Hong Kong - Hang Seng',
    '^AXJO': 'Australia - S&P/ASX 200',
    '^NSEI': 'India - Nifty 50',
    '^GSPTSE': 'Canada - S&P/TSX Composite',
    '^GDAXI': 'Germany - DAX',
    '^FTSE': 'United Kingdom - FTSE 100',
    '^N225': 'Japan - Nikkei 225',
    '^MXX': 'Mexico - IPC',
    '^BVSP': 'Brazil - Ibovespa'
}

prices = yf.download(
    list(indexes), start='2026-01-01', end='2026-08-22',
    auto_adjust=True, progress=False,
)['Close']

ytd = pd.DataFrame({
    'index': [indexes[t] for t in prices.columns],
    'first_close': [prices[t].dropna().iloc[0] for t in prices.columns],
    'last_close': [prices[t].dropna().iloc[-1] for t in prices.columns],
}, index=prices.columns)
ytd['ytd_return_%'] = (ytd['last_close'] / ytd['first_close'] - 1) * 100
ytd = ytd.sort_values('ytd_return_%', ascending=False)

ytd


,index,first_close,last_close,ytd_return_%
Ticker,,,,
^N225,Japan - Nikkei 225,51832.800781,66016.359375,27.364060
^GSPTSE,Canada - S&P/TSX Composite,31883.400391,36620.199219,14.856630
^GSPC,United States - S&P 500,6858.470215,7674.370117,11.896237
^FTSE,United Kingdom - FTSE 100,9951.099609,10816.599609,8.697531
^BVSP,Brazil - Ibovespa,160539.000000,171032.000000,6.536106
^GDAXI,Germany - DAX,24539.339844,26136.560547,6.508817
^AXJO,Australia - S&P/ASX 200,8727.799805,9058.900391,3.793632
^MXX,Mexico - IPC,64141.359375,65729.179688,2.475501
^HSI,Hong Kong - Hang Seng,26338.470703,26009.460938,-1.249160


In [6]:
us = ytd.loc['^GSPC', 'ytd_return_%']
better = ytd[ytd['ytd_return_%'] > us]
better['index']


Ticker
^N225              Japan - Nikkei 225
^GSPTSE    Canada - S&P/TSX Composite
Name: index, dtype: str

## 3.

[Index] S&P 500 Market Corrections Analysis
Calculate the median drawdown (in %) of significant market corrections in the S&P 500 index.

For this task, define a correction as an event when a stock index goes down by at least 5% from the most recent all-time high.

Steps:

- Download S&P 500 historical data (period from 1950 to present) using yfinance (daily stats)
- Identify all-time high closing price points (where price exceeds all previous days' prices)
- For each pair of consecutive all-time highs, find the minimum price in between
- Calculate drawdown percentages: (high - low) / high × 100
- Filter for corrections with at least 5% drawdown
- Calculate the duration in days for each correction period
- Determine the 25th, 50th (median), and 75th percentiles for correction durations and drawdowns

In [7]:
spx = yf.Ticker('^GSPC').history(start='1950-01-01', interval='1d')
close = spx['Close']

# all-time high
is_ath = close == close.cummax()
ath_pos = np.flatnonzero(is_ath.to_numpy())

values = close.to_numpy()
dates = close.index

rows = []
for start_pos, end_pos in zip(ath_pos[:-1], ath_pos[1:]):
    if end_pos - start_pos < 2:
        continue
    segment = values[start_pos + 1:end_pos]
    trough_pos = start_pos + 1 + segment.argmin()
    high = values[start_pos]
    low = values[trough_pos]
    rows.append({
        'high_date': dates[start_pos],
        'high': high,
        'trough_date': dates[trough_pos],
        'low': low,
        'recovery_date': dates[end_pos],
        'drawdown_%': (high - low) / high * 100,
        'duration_days': (dates[end_pos] - dates[start_pos]).days,
    })

drawdowns = pd.DataFrame(rows)
corrections = drawdowns[drawdowns['drawdown_%'] >= 5].reset_index(drop=True)


In [8]:
corrections[['drawdown_%', 'duration_days']].quantile([0.25, 0.5, 0.75])


,drawdown_%,duration_days
0.25,6.234677,55.50
0.50,7.986358,92.50
0.75,14.019826,211.25


## 4. 

[Stocks] Earnings Surprise Analysis for Amazon (AMZN)

Calculate the median 2-day percentage change in stock prices following positive earnings surprise days.

- Download complete historical price data using yfinance
- Calculate 2-day percentage changes for all historical dates: for each sequence of 3 consecutive trading days (Day 1, Day 2, Day 3), compute the return as Close_Day3 / Close_Day1 - 1. (Assume Day 2 may correspond to the earnings announcement.)
- Filter for positive earnings surprises and calculate the median 2-day return. Then calculate the correlation between the 2-day stock return and the earnings surprise magnitude.

In [9]:
ticker = 'AMZN'
ticker_obj = yf.Ticker(ticker)

earnings = ticker_obj.get_earnings_dates(limit=100)
earnings = earnings[~earnings.index.duplicated()].sort_index()

In [10]:
hist = ticker_obj.history(period='max', interval='1d')
close = hist['Close']

# ---------------------------------------
ret_2d = (close.shift(-1) / close.shift(1) - 1)
ret_2d.index = ret_2d.index.date

# ---------------------------------------
surprises = earnings[['EPS Estimate', 'Reported EPS', 'Surprise(%)']].copy()
surprises.index = surprises.index.date
surprises = surprises[~surprises.index.duplicated()]

surprises['ret_2d_%'] = pd.Series(ret_2d).reindex(surprises.index) * 100

# ---------------------------------------
positive = surprises[(surprises['Surprise(%)'] > 0) & surprises['ret_2d_%'].notna()]

positive['ret_2d_%'].median()

np.float64(1.8952678440074866)